In [1]:
import json
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name: str = "LazarusNLP/all-indo-e5-small-v4"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        passages = [f"passage: {text.strip()}" for text in texts]
        return self.model.encode(
            passages,
            batch_size=32,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,  # Supaya kelihatan progresnya di Notebook
        ).tolist()

    def embed_query(self, text):
        query = f"query: {text.strip()}"
        return self.model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )[0].tolist()


c:\Users\Ammar\miniconda3\envs\scraping\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print("📚 Inisialisasi Model Lazarus...")
embeddings = SentenceTransformerEmbeddings("LazarusNLP/all-indo-e5-small-v4")

# ========================================================
# KITA BUAT COLLECTION BARU SESUAI PERMINTAAN
collection_name = "embed-v1" 
# ========================================================

client = QdrantClient(host="localhost", port=6333)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name, # Menunjuk ke collection yang baru
    embedding=embeddings,
)

file_path = "ChunksBindo.json" # Pastikan file JSON sejajar dengan file notebook ini
print(f"Membaca data chunk dari: {file_path}")
with open(file_path, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

# Proses menjadi objek Document (Langchain format)
documents = []
for item in chunks_data:
    text = item.get("text", "")
    metadata = item.get("metadata", {})
    doc = Document(page_content=text, metadata=metadata)
    documents.append(doc)

print(f"✅ Berhasil meload {len(documents)} chunk dari file JSON.")


📚 Inisialisasi Model Lazarus...


UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `bindo_rag_collection_v1` doesn\'t exist!"},"time":0.000504838}'

In [3]:
import json
import os
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

# 1. DEFINE EMBEDDING CLASS
class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name: str = "LazarusNLP/all-indo-e5-small-v4"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        passages = [f"passage: {text.strip()}" for text in texts]
        return self.model.encode(passages, batch_size=32, normalize_embeddings=True, show_progress_bar=True).tolist()

    def embed_query(self, text):
        query = f"query: {text.strip()}"
        return self.model.encode([query], normalize_embeddings=True)[0].tolist()

# 2. SETUP CONFIG
collection_name = "bindo_rag_collection_v1"
file_path = "ChunksBindo.json"

print("📚 Inisialisasi Model Lazarus...")
embeddings = SentenceTransformerEmbeddings("LazarusNLP/all-indo-e5-small-v4")
client = QdrantClient(host="localhost", port=6333)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

# 3. LOAD DATA
print(f"📂 Membaca file: {file_path}")
if not os.path.exists(file_path):
    print(f"❌ ERROR: File {file_path} tidak ditemukan! Pastikan file JSON ada di folder yang sama dengan notebook ini.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        chunks_data = json.load(f)

    documents = [Document(page_content=item["text"], metadata=item["metadata"]) for item in chunks_data]
    print(f"✅ Berhasil meload {len(documents)} chunk.")

    # 4. INGEST
    batch_size = 100
    print(f"🚀 Memasukkan data ke Qdrant (Collection: {collection_name})...")
    for i in range(0, len(documents), batch_size):
        batch_docs = documents[i : i + batch_size]
        vectorstore.add_documents(batch_docs)
        print(f"✔️ Batch {int(i/batch_size)+1} selesai...")

    print("🎉 SEMUA SELESAI!")


📚 Inisialisasi Model Lazarus...


UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `bindo_rag_collection_v1` doesn\'t exist!"},"time":0.000073046}'

In [ ]:
batch_size = 100
total_batches = (len(documents) + batch_size - 1) // batch_size

print(f"🚀 Memulai proses ingest (embedding) ke dalam collection '{collection_name}'...")

for i in range(0, len(documents), batch_size):
    batch_docs = documents[i : i + batch_size]
    
    # Perintah ini akan mengirim data ke Qdrant, dan otomatis 
    # membuat collection_name baru (bindo_rag_collection_v1) jika belum ada
    vectorstore.add_documents(batch_docs)
    
    current_batch = (i // batch_size) + 1
    print(f"✅ Ingest Batch {current_batch}/{total_batches} sukses! (chunk ke-{i} s/d {i + len(batch_docs) - 1})")

print("🎉 Selesai! Semua chunk Bahasa Indonesia sudah termuat murni di collection baru.")
